In [3]:
from src.agent import HelpDeskAgent
from src.configs import Settings
from src.tools.search_xyz_manual import search_xyz_manual
from src.tools.search_xyz_qa import (
    search_xyz_qa,
)

settings = Settings()


In [4]:
agent = HelpDeskAgent(
    settings=settings,
    tools=[search_xyz_manual, search_xyz_qa],
)

In [5]:
# question = """
# お世話になっております。

# 現在、XYZシステムの利用を検討しており、以下の2点についてご教示いただければと存じます。

# 1. パスワードに利用可能な文字の制限について
# 当該システムにてパスワードを設定する際、使用可能な文字の範囲（例：英数字、記号、文字数制限など）について詳しい情報をいただけますでしょうか。安全かつシステムでの認証エラーを防ぐため、具体的な仕様を確認したいと考えております。

# 2. 最新リリースの取得方法について
# 最新のアップデート情報をどのように確認・取得できるかについてもお教えいただけますと幸いです。

# お忙しいところ恐縮ですが、ご対応のほどよろしくお願い申し上げます。
# """

question = """
お世話になっております。

現在、XYZシステムを利用を検討しており、以下の点についてご教示いただければと存じます。

1. 特定のプロジェクトに対してのみ通知を制限する方法について

2. パスワードに利用可能な文字の制限について
当該システムにてパスワードを設定する際、使用可能な文字の範囲（例：英数字、記号、文字数制限など）について詳しい情報をいただけますでしょうか。安全かつシステムでの認証エラーを防ぐため、具体的な仕様を確認したいと考えております。

お忙しいところ恐縮ですが、ご対応のほどよろしくお願い申し上げます。

"""

## 計画ステップ

In [6]:
input_data = {"question": question}

plan_result = agent.create_plan(state=input_data)

2026-04-04 16:06:25,999 INFO 🚀 Starting plan generation process...
2026-04-04 16:06:26,000 INFO Sending request to OpenAI...
2026-04-04 16:06:28,481 INFO HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-04 16:06:28,501 INFO ✅ Successfully received response from OpenAI.
2026-04-04 16:06:28,502 INFO Plan generation complete!


In [7]:
plan_result["plan"]

['XYZシステムで特定のプロジェクトに対してのみ通知を制限する方法について調べる',
 'XYZシステムのパスワード設定における使用可能な文字の範囲（英数字、記号、文字数制限など）について調べる']

## ツール選択ステップ

In [8]:
input_data = {
    "question": question,
    "plan": plan_result["plan"],
    "subtask": plan_result["plan"][0],
    "challenge_count": 0,
    "is_completed": False,
}

In [26]:
select_tool_result = agent.select_tools(state=input_data)

2026-04-04 16:13:40,124 INFO 🚀 Starting tool selection process...
2026-04-04 16:13:40,133 INFO Sending request to OpenAI...
2026-04-04 16:13:41,615 INFO HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-04 16:13:41,620 INFO ✅ Successfully received response from OpenAI.
2026-04-04 16:13:41,620 INFO Tool selection complete!


In [27]:
select_tool_result

{'messages': [{'role': 'system',
   'content': '\nあなたはXYZというシステムの質問応答のためにサブタスク実行を担当するエージェントです。\n回答までの全体の流れは計画立案 → サブタスク実行 [ツール実行 → サブタスク回答 → リフレクション] → 最終回答となります。\nサブタスクはユーザーの質問に回答するために考えられた計画の一つです。\n最終的な回答は全てのサブタスクの結果を組み合わせて別エージェントが作成します。\nあなたは以下の1~3のステップを指示に従ってそれぞれ実行します。各ステップは指示があったら実行し、同時に複数ステップの実行は行わないでください。\nなおリフレクションの結果次第で所定の回数までツール選択・実行を繰り返します。\n\n1. ツール選択・実行\nサブタスク回答のためのツール選択と選択されたツールの実行を行います。\n2回目以降はリフレクションのアドバイスに従って再実行してください。\n\n2. サブタスク回答\nツールの実行結果はあなたしか観測できません。\nツールの実行結果から得られた回答に必要なことは言語化し、最後の回答用エージェントに引き継げるようにしてください。\n例えば、概要を知るサブタスクならば、ツールの実行結果から概要を言語化してください。\n手順を知るサブタスクならば、ツールの実行結果から手順を言語化してください。\n回答できなかった場合は、その旨を言語化してください。\n\n3. リフレクション\nツールの実行結果と回答から、サブタスクに対して正しく回答できているかを評価します。\n回答がわからない、情報が見つからないといった内容の場合は評価をNGにし、やり直すようにしてください。\n評価がNGの場合は、別のツールを試す、別の文言でツールを試すなど、なぜNGなのかとどうしたら改善できるかを考えアドバイスを作成してください。\nアドバイスの内容は過去のアドバイスと計画内の他のサブタスクと重複しないようにしてください。\nアドバイスの内容をもとにツール選択・実行からやり直します。\n評価がOKの場合は、サブタスク回答を終了します。\n\n'},
  {'role': 'user',
   'content': "\nユーザーの元の質問: \nお世話になっております。\

In [11]:
select_tool_result["messages"][-1]

{'role': 'assistant',
 'tool_calls': [{'id': 'call_9yekHiCorBwrZRuC3nKIX338',
   'function': {'arguments': '{"keywords":"特定のプロジェクト 通知制限"}',
    'name': 'search_xyz_manual'},
   'type': 'function'}]}

In [12]:
select_tool_result["messages"]

[{'role': 'system',
  'content': '\nあなたはXYZというシステムの質問応答のためにサブタスク実行を担当するエージェントです。\n回答までの全体の流れは計画立案 → サブタスク実行 [ツール実行 → サブタスク回答 → リフレクション] → 最終回答となります。\nサブタスクはユーザーの質問に回答するために考えられた計画の一つです。\n最終的な回答は全てのサブタスクの結果を組み合わせて別エージェントが作成します。\nあなたは以下の1~3のステップを指示に従ってそれぞれ実行します。各ステップは指示があったら実行し、同時に複数ステップの実行は行わないでください。\nなおリフレクションの結果次第で所定の回数までツール選択・実行を繰り返します。\n\n1. ツール選択・実行\nサブタスク回答のためのツール選択と選択されたツールの実行を行います。\n2回目以降はリフレクションのアドバイスに従って再実行してください。\n\n2. サブタスク回答\nツールの実行結果はあなたしか観測できません。\nツールの実行結果から得られた回答に必要なことは言語化し、最後の回答用エージェントに引き継げるようにしてください。\n例えば、概要を知るサブタスクならば、ツールの実行結果から概要を言語化してください。\n手順を知るサブタスクならば、ツールの実行結果から手順を言語化してください。\n回答できなかった場合は、その旨を言語化してください。\n\n3. リフレクション\nツールの実行結果と回答から、サブタスクに対して正しく回答できているかを評価します。\n回答がわからない、情報が見つからないといった内容の場合は評価をNGにし、やり直すようにしてください。\n評価がNGの場合は、別のツールを試す、別の文言でツールを試すなど、なぜNGなのかとどうしたら改善できるかを考えアドバイスを作成してください。\nアドバイスの内容は過去のアドバイスと計画内の他のサブタスクと重複しないようにしてください。\nアドバイスの内容をもとにツール選択・実行からやり直します。\n評価がOKの場合は、サブタスク回答を終了します。\n\n'},
 {'role': 'user',
  'content': "\nユーザーの元の質問: \nお世話になっております。\n\n現在、XYZシステムを利用

## ツール実行ステップ

In [13]:
input_data = {
    "question": question,
    "plan": plan_result["plan"],
    "subtask": plan_result["plan"][0],
    "challenge_count": 0,
    "messages": select_tool_result["messages"],
    "is_completed": False,
}

In [14]:
tool_results = agent.execute_tools(state=input_data)

2026-04-04 16:08:15,729 INFO 🚀 Starting tool execution process...
2026-04-04 16:08:15,733 INFO Searching XYZ manual by keyword: {"keywords":"特定のプロジェクト 通知制限"}
2026-04-04 16:08:15,776 INFO POST http://localhost:9200/documents/_search [status:200 duration:0.042s]
2026-04-04 16:08:15,777 INFO Search results: 10 hits
2026-04-04 16:08:15,777 INFO Finished searching XYZ manual by keyword
2026-04-04 16:08:15,777 INFO Tool execution complete!


In [15]:
tool_results["tool_results"][0][0].results

[SearchOutput(file_name='XYZシステムリリースノート.pdf', content='影響範囲 :\n管理者とプロジェクトマネージャー。リソース管理が強化され、プロジェクトの効\n率化に寄与。\nv 7 . 0  -  バックアップ監視と通知機能\n概要 :\nバックアップの状態をリアルタイムで監視し、バックアップ失敗時には即座にメー\nルや SMS で通知を受け取ることができる機能を追加しました。以前のシステムで\nは、バックアップ失敗の際に即時対応が難しく、結果としてデータ損失のリスクが\n⾼まりました。この機能により、バックアップの状態を常に把握し、迅速に対応す\nることが可能になりました。\nさらに、通知チャネルの設定が可能で、特定のイベントに基づいたアラート通知を'),
 SearchOutput(file_name='XYZシステムリリースノート.pdf', content='バグ修正 :\nプロジェクトメンバーの追加が特定の状況で失敗する問題を修正しました。\n影響範囲 :\nプロジェクト管理者。複数プロジェクト間でのリソース共有およびタスク管理が容\n易に。\nv 5 . 0  -  レポート作成とメール配信機能\n概要 :\nカスタムレポート作成機能を追加し、ユーザーが必要に応じて特定のデータソース\nを選択してレポートを作成し、カスタマイズできるようにしました。従来、レポー\nト作成の柔軟性が不⾜しており、ユーザーは定型レポートに頼らざるを得ない状況\nでしたが、この改善により、ビジネスニーズに応じた⾃由度の⾼いレポート作成が\n可能となりました。'),
 SearchOutput(file_name='XYZシステムリリースノート.pdf', content='改善により、プロジェクトの⽴ち上げにかかる負担を軽減しました。\n影響範囲 :\n全ユーザー。アカウント管理機能とプロジェクト作成の効率が向上。\nv0 . 0 \ue088 初期リリース\nv1. 0 \ue088 基本機能の実装\nv2. 0 \ue088 データセキュリティとバックアップ\nv3 . 0 \ue088 ユーザー安全性の向上\nv4 . 0 \ue088 プロジェクト管理機能の改善\nv5 . 0 \ue088 レポート作成とメール配信機能\n

In [16]:
tool_results

{'messages': [{'role': 'system',
   'content': '\nあなたはXYZというシステムの質問応答のためにサブタスク実行を担当するエージェントです。\n回答までの全体の流れは計画立案 → サブタスク実行 [ツール実行 → サブタスク回答 → リフレクション] → 最終回答となります。\nサブタスクはユーザーの質問に回答するために考えられた計画の一つです。\n最終的な回答は全てのサブタスクの結果を組み合わせて別エージェントが作成します。\nあなたは以下の1~3のステップを指示に従ってそれぞれ実行します。各ステップは指示があったら実行し、同時に複数ステップの実行は行わないでください。\nなおリフレクションの結果次第で所定の回数までツール選択・実行を繰り返します。\n\n1. ツール選択・実行\nサブタスク回答のためのツール選択と選択されたツールの実行を行います。\n2回目以降はリフレクションのアドバイスに従って再実行してください。\n\n2. サブタスク回答\nツールの実行結果はあなたしか観測できません。\nツールの実行結果から得られた回答に必要なことは言語化し、最後の回答用エージェントに引き継げるようにしてください。\n例えば、概要を知るサブタスクならば、ツールの実行結果から概要を言語化してください。\n手順を知るサブタスクならば、ツールの実行結果から手順を言語化してください。\n回答できなかった場合は、その旨を言語化してください。\n\n3. リフレクション\nツールの実行結果と回答から、サブタスクに対して正しく回答できているかを評価します。\n回答がわからない、情報が見つからないといった内容の場合は評価をNGにし、やり直すようにしてください。\n評価がNGの場合は、別のツールを試す、別の文言でツールを試すなど、なぜNGなのかとどうしたら改善できるかを考えアドバイスを作成してください。\nアドバイスの内容は過去のアドバイスと計画内の他のサブタスクと重複しないようにしてください。\nアドバイスの内容をもとにツール選択・実行からやり直します。\n評価がOKの場合は、サブタスク回答を終了します。\n\n'},
  {'role': 'user',
   'content': "\nユーザーの元の質問: \nお世話になっております。\

## サブタスク回答

In [17]:
input_data = {
    "question": question,
    "plan": plan_result["plan"],
    "subtask": plan_result["plan"][0],
    "challenge_count": 0,
    "messages": tool_results["messages"],
    "tool_results": tool_results["tool_results"],
    "is_completed": False,
}

In [18]:
subtask_answer = agent.create_subtask_answer(state=input_data)

2026-04-04 16:09:03,303 INFO 🚀 Starting subtask answer creation process...
2026-04-04 16:09:03,304 INFO Sending request to OpenAI...
2026-04-04 16:09:07,346 INFO HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-04 16:09:07,350 INFO ✅ Successfully received response from OpenAI.
2026-04-04 16:09:07,351 INFO Subtask answer creation complete!


In [19]:
subtask_answer

{'messages': [{'role': 'system',
   'content': '\nあなたはXYZというシステムの質問応答のためにサブタスク実行を担当するエージェントです。\n回答までの全体の流れは計画立案 → サブタスク実行 [ツール実行 → サブタスク回答 → リフレクション] → 最終回答となります。\nサブタスクはユーザーの質問に回答するために考えられた計画の一つです。\n最終的な回答は全てのサブタスクの結果を組み合わせて別エージェントが作成します。\nあなたは以下の1~3のステップを指示に従ってそれぞれ実行します。各ステップは指示があったら実行し、同時に複数ステップの実行は行わないでください。\nなおリフレクションの結果次第で所定の回数までツール選択・実行を繰り返します。\n\n1. ツール選択・実行\nサブタスク回答のためのツール選択と選択されたツールの実行を行います。\n2回目以降はリフレクションのアドバイスに従って再実行してください。\n\n2. サブタスク回答\nツールの実行結果はあなたしか観測できません。\nツールの実行結果から得られた回答に必要なことは言語化し、最後の回答用エージェントに引き継げるようにしてください。\n例えば、概要を知るサブタスクならば、ツールの実行結果から概要を言語化してください。\n手順を知るサブタスクならば、ツールの実行結果から手順を言語化してください。\n回答できなかった場合は、その旨を言語化してください。\n\n3. リフレクション\nツールの実行結果と回答から、サブタスクに対して正しく回答できているかを評価します。\n回答がわからない、情報が見つからないといった内容の場合は評価をNGにし、やり直すようにしてください。\n評価がNGの場合は、別のツールを試す、別の文言でツールを試すなど、なぜNGなのかとどうしたら改善できるかを考えアドバイスを作成してください。\nアドバイスの内容は過去のアドバイスと計画内の他のサブタスクと重複しないようにしてください。\nアドバイスの内容をもとにツール選択・実行からやり直します。\n評価がOKの場合は、サブタスク回答を終了します。\n\n'},
  {'role': 'user',
   'content': "\nユーザーの元の質問: \nお世話になっております。\

In [20]:
print(subtask_answer["subtask_answer"])

1. ツール選択・実行
XYZシステムのマニュアルを検索しましたが、特定のプロジェクトに対してのみ通知を制限する方法に関する具体的な情報は見つかりませんでした。

2. サブタスク回答
XYZシステムで特定のプロジェクトに対してのみ通知を制限する方法についての情報は、現在の検索結果からは見つかりませんでした。

3. リフレクション
評価: NG
理由: 特定のプロジェクトに対してのみ通知を制限する方法についての具体的な情報が見つかりませんでした。

アドバイス: 
- 別のキーワードで再度検索を試みる。
- XYZシステムの公式サポートやフォーラムでの情報を探す。
- 通知設定に関する一般的なガイドラインを確認する。

次に、別のキーワードで再度検索を試みます。


## リフレクション

In [21]:
input_data = {
    "question": question,
    "plan": plan_result["plan"],
    "subtask": plan_result["plan"][0],
    "challenge_count": 0,
    "messages": subtask_answer["messages"],
    "tool_results": tool_results["tool_results"],
    "is_completed": False,
    "subtask_answer": subtask_answer["subtask_answer"],
}

In [22]:
reflection_result = agent.reflect_subtask(state=input_data)

2026-04-04 16:09:57,418 INFO 🚀 Starting reflection process...
2026-04-04 16:09:57,418 INFO Sending request to OpenAI...
2026-04-04 16:09:59,504 INFO HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-04 16:09:59,507 INFO ✅ Successfully received response from OpenAI.
2026-04-04 16:09:59,507 INFO Reflection complete!


In [23]:
reflection_result

{'messages': [{'role': 'system',
   'content': '\nあなたはXYZというシステムの質問応答のためにサブタスク実行を担当するエージェントです。\n回答までの全体の流れは計画立案 → サブタスク実行 [ツール実行 → サブタスク回答 → リフレクション] → 最終回答となります。\nサブタスクはユーザーの質問に回答するために考えられた計画の一つです。\n最終的な回答は全てのサブタスクの結果を組み合わせて別エージェントが作成します。\nあなたは以下の1~3のステップを指示に従ってそれぞれ実行します。各ステップは指示があったら実行し、同時に複数ステップの実行は行わないでください。\nなおリフレクションの結果次第で所定の回数までツール選択・実行を繰り返します。\n\n1. ツール選択・実行\nサブタスク回答のためのツール選択と選択されたツールの実行を行います。\n2回目以降はリフレクションのアドバイスに従って再実行してください。\n\n2. サブタスク回答\nツールの実行結果はあなたしか観測できません。\nツールの実行結果から得られた回答に必要なことは言語化し、最後の回答用エージェントに引き継げるようにしてください。\n例えば、概要を知るサブタスクならば、ツールの実行結果から概要を言語化してください。\n手順を知るサブタスクならば、ツールの実行結果から手順を言語化してください。\n回答できなかった場合は、その旨を言語化してください。\n\n3. リフレクション\nツールの実行結果と回答から、サブタスクに対して正しく回答できているかを評価します。\n回答がわからない、情報が見つからないといった内容の場合は評価をNGにし、やり直すようにしてください。\n評価がNGの場合は、別のツールを試す、別の文言でツールを試すなど、なぜNGなのかとどうしたら改善できるかを考えアドバイスを作成してください。\nアドバイスの内容は過去のアドバイスと計画内の他のサブタスクと重複しないようにしてください。\nアドバイスの内容をもとにツール選択・実行からやり直します。\n評価がOKの場合は、サブタスク回答を終了します。\n\n'},
  {'role': 'user',
   'content': "\nユーザーの元の質問: \nお世話になっております。\

In [24]:
# 最初に選択されたツールを確認
print(reflection_result["messages"][2]["tool_calls"][0]["function"]["name"])

search_xyz_manual


In [25]:
# リフレクション結果の確認
print("is_completed =", reflection_result["reflection_results"][0].is_completed)
print("advice =", reflection_result["reflection_results"][0].advice)

is_completed = False
advice = 別のキーワードで再度検索を試みる。例えば、「通知設定」「プロジェクト別通知」などのキーワードを使用してみる。また、XYZシステムの公式サポートやフォーラムでの情報を探すことも考慮する。
